# 🧠 RLVR: Reinforcement Learning with Verifiable Rewards
## End-to-End GRPO Training on GSM8K — Kaggle T4 GPU

This notebook trains a **Qwen2.5-1.5B-Instruct** model to solve math word problems using **GRPO (Group Relative Policy Optimization)** — the same algorithm used to train DeepSeek-R1.

### What You'll Learn
1. **Data Preparation** — Format GSM8K for GRPO training
2. **Reward Engineering** — Build correctness and format verifiers
3. **GRPO Training** — Configure and run the training loop
4. **Evaluation** — Compare before vs. after performance

### Requirements
- **GPU**: T4 (16GB VRAM) — Select under Settings → Accelerator
- **Runtime**: ~2-3 hours for 250 training steps
- **Internet**: Required for model and dataset downloads

---
## 🔧 Section 0: Environment Setup

In [ ]:
# Install required packages
# Unsloth provides memory-efficient model loading & LoRA
# TRL provides GRPOTrainer
!pip install -q --upgrade unsloth trl datasets transformers accelerate

In [ ]:
# Pin to single GPU (cuda:0) to prevent multi-device sharding conflicts on Kaggle T4 x2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Verify GPU is available and compatible
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"Active GPU:      {device_name} (Compute Capability {cap[0]}.{cap[1]})")
    print(f"VRAM:            {vram:.1f} GB")
    
    if cap[0] < 7:
        print("\n" + "="*70)
        print(f"❌ INCOMPATIBLE GPU DETECTED: {device_name} (Compute Capability {cap[0]}.{cap[1]})")
        print("Unsloth and modern Triton kernels require Compute Capability >= 7.0 (e.g. T4, A100, L4).")
        print("Tesla P100 (Pascal, 6.0) is not supported and causes cudaErrorNoKernelImageForDevice.")
        print("\n👉 ACTION REQUIRED ON KAGGLE:")
        print("   1. Look at the right sidebar under Notebook options / Settings")
        print("   2. Under Accelerator, change from GPU P100 to GPU T4 x2 (or GPU T4)")
        print("   3. Restart session and re-run.")
        print("="*70 + "\n")
        raise RuntimeError("Incompatible GPU: Please switch Kaggle Accelerator from P100 to GPU T4 x2 or GPU T4 in Settings.")
    else:
        print("✅ Single GPU pinned & architecture is compatible (Compute Capability >= 7.0)!")
else:
    print("⚠️  No GPU detected! Enable GPU in Settings → Accelerator → GPU T4 x2")


---
## 📊 Section 1: Data Preparation

RLVR needs data with **verifiable answers** — we use GSM8K (Grade School Math 8K).

Each sample has:
- A math word problem (the prompt)
- A numerical ground-truth answer (for the verifier)

The model will learn to generate its own reasoning chains.

In [ ]:
import re
from datasets import load_dataset

# ═══════════════════════════════════════════════════════════════════
# System Prompt — Teaches the model HOW to respond
# ═══════════════════════════════════════════════════════════════════
# The XML tags (<reasoning>, <answer>) serve two purposes:
# 1. Force the model to think before answering
# 2. Make it easy for the verifier to extract the final answer

SYSTEM_PROMPT = """You are a helpful math assistant. Solve problems step by step.

Think through the problem inside <reasoning> tags.
Provide your final numerical answer inside <answer> tags.

<reasoning>
Your step-by-step reasoning here.
</reasoning>
<answer>numerical answer only</answer>"""

print("System prompt defined ✅")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Helper Functions with Examples
# ═══════════════════════════════════════════════════════════════════

def extract_gsm8k_answer(answer_text: str) -> str:
    """Extract the final numerical answer from GSM8K ground truth text.
    
    GSM8K answers follow 'reasoning text #### <answer>'. This function
    removes reasoning prefix, currency symbols ($), and commas (,).
    
    Args:
        answer_text: Raw answer string from dataset.
        
    Returns:
        Cleaned numerical answer string.
        
    Examples:
        >>> extract_gsm8k_answer('Janet sells 9 eggs at $2 each. #### 18')
        '18'
        >>> extract_gsm8k_answer('Total profit is #### $1,250')
        '1250'
    """
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        answer = match.group(1).strip()
        answer = answer.replace(",", "").replace("$", "")
        return answer
    return answer_text.strip()


def format_prompt(question: str) -> list:
    """Convert a raw question into chat-formatted system and user messages.
    
    Args:
        question: Math word problem text.
        
    Returns:
        List of message dicts: [system_message, user_message].
        
    Examples:
        >>> msgs = format_prompt('What is 15 + 27?')
        >>> len(msgs)
        2
        >>> msgs[1]['content']
        'What is 15 + 27?'
    """
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


# Quick test
test_answer = "She makes 9 * 2 = <<9*2=18>>$18.\n#### 18"
print(f"Raw:       {test_answer}")
print(f"Extracted: {extract_gsm8k_answer(test_answer)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Load and Format GSM8K
# ═══════════════════════════════════════════════════════════════════

# Load dataset from Hugging Face
raw_train = load_dataset("openai/gsm8k", "main", split="train")
raw_test = load_dataset("openai/gsm8k", "main", split="test")

print(f"Raw train: {len(raw_train)} samples")
print(f"Raw test:  {len(raw_test)} samples")

# Show a raw sample
print(f"\n--- Raw Sample ---")
print(f"Question: {raw_train[0]['question'][:200]}")
print(f"Answer:   {raw_train[0]['answer'][:200]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Format for GRPOTrainer
# ═══════════════════════════════════════════════════════════════════
# GRPOTrainer expects:
#   - 'prompt': list[dict] — chat-format messages
#   - 'solution': str — ground truth (passed to reward functions via kwargs)

# Use 500 training samples (faster, fits in Kaggle session)
MAX_TRAIN_SAMPLES = 500
MAX_TEST_SAMPLES = 100

train_dataset = raw_train.select(range(MAX_TRAIN_SAMPLES))
test_dataset = raw_test.select(range(MAX_TEST_SAMPLES))

def transform(example):
    return {
        "prompt": format_prompt(example["question"]),
        "solution": extract_gsm8k_answer(example["answer"]),
    }

train_dataset = train_dataset.map(transform, remove_columns=raw_train.column_names)
test_dataset = test_dataset.map(transform, remove_columns=raw_test.column_names)

print(f"Formatted train: {len(train_dataset)} samples")
print(f"Formatted test:  {len(test_dataset)} samples")

# Verify a formatted sample
sample = train_dataset[0]
print(f"\n--- Formatted Sample ---")
print(f"Prompt (system): {sample['prompt'][0]['content'][:60]}...")
print(f"Prompt (user):   {sample['prompt'][1]['content'][:80]}...")
print(f"Solution:        {sample['solution']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Validate the Dataset
# ═══════════════════════════════════════════════════════════════════

issues = []
for i in range(len(train_dataset)):
    s = train_dataset[i]
    # Check structure
    assert len(s["prompt"]) == 2, f"Sample {i}: wrong prompt length"
    assert s["prompt"][0]["role"] == "system"
    assert s["prompt"][1]["role"] == "user"
    # Check answer is numeric
    try:
        float(s["solution"])
    except ValueError:
        issues.append(f"Sample {i}: non-numeric answer '{s['solution']}'")

if issues:
    for issue in issues:
        print(f"⚠️  {issue}")
else:
    print(f"✅ All {len(train_dataset)} training samples passed validation!")

---
## 🎯 Section 2: Reward Functions (Verifiers)

RLVR replaces learned reward models with **deterministic verifiers**.

We define two reward functions:
1. **Correctness Reward** — Does the answer match ground truth? (0.0 or 1.0)
2. **Format Reward** — Does the output use proper XML tags? (0.0, 0.5, or 1.0)

These are summed to give a total reward of 0.0 to 2.0 per completion.

**Important**: TRL's GRPOTrainer passes `prompts` and `completions` as the first two positional arguments, and any extra dataset columns (like `solution`) via `**kwargs`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Robust Text Extraction Helper
# ═══════════════════════════════════════════════════════════════════

def extract_completion_text(completion) -> str:
    """Extract clean assistant text whether completion is a string, dict, or message list.
    
    TRL GRPOTrainer passes completions as a list of message dicts
    (e.g., [{'role': 'assistant', 'content': '...'}]) in chat template mode.
    
    Args:
        completion: Raw completion object (str, list[dict], or dict).
        
    Returns:
        Extracted string content.
        
    Examples:
        >>> extract_completion_text('<answer>42</answer>')
        '<answer>42</answer>'
        >>> extract_completion_text([{'role': 'assistant', 'content': '<answer>18</answer>}])
        '<answer>18</answer>'
    """
    if isinstance(completion, str):
        return completion
    elif isinstance(completion, list):
        if len(completion) > 0:
            if isinstance(completion[-1], dict) and "content" in completion[-1]:
                return completion[-1]["content"]
            elif isinstance(completion[0], dict) and "content" in completion[0]:
                return completion[0]["content"]
            return str(completion[-1])
        return ""
    elif isinstance(completion, dict) and "content" in completion:
        return completion["content"]
    return str(completion)


# ═══════════════════════════════════════════════════════════════════
# Correctness Reward Verifier
# ═══════════════════════════════════════════════════════════════════

def correctness_reward(prompts, completions, solution, **kwargs):
    """Score answer correctness against ground truth within 1e-4 tolerance.
    
    Args:
        prompts: List of input prompts from GRPOTrainer.
        completions: List of model-generated text or message dicts.
        solution: List of ground-truth answers from dataset column.
        **kwargs: Additional dataset columns passed via GRPOTrainer.
        
    Returns:
        List[float]: 1.0 for correct answer, 0.0 for incorrect.
        
    Examples:
        >>> prompts = ['Problem 1']
        >>> comp = ['<reasoning>9*2=18</reasoning><answer>18</answer>']
        >>> correctness_reward(prompts, comp, solution=['18'])
        [1.0]
        >>> comp_money = ['<reasoning>Calc</reasoning><answer>$18.00</answer>']
        >>> correctness_reward(prompts, comp_money, solution=['18'])
        [1.0]
    """
    rewards = []
    for completion, sol in zip(completions, solution):
        text = extract_completion_text(completion)
        match = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.DOTALL)
        if match:
            model_ans = match.group(1).strip().replace(",", "").replace("$", "")
            sol_clean = str(sol).strip().replace(",", "").replace("$", "")
            try:
                rewards.append(1.0 if abs(float(model_ans) - float(sol_clean)) < 1e-4 else 0.0)
            except ValueError:
                rewards.append(1.0 if model_ans == sol_clean else 0.0)
        else:
            rewards.append(0.0)
    return rewards


# ═══════════════════════════════════════════════════════════════════
# Format Reward Verifier
# ═══════════════════════════════════════════════════════════════════

def format_reward(prompts, completions, **kwargs):
    """Score XML tag structure compliance.
    
    Scoring:
        - 1.0: Both <reasoning> and <answer> tags present.
        - 0.5: Only <answer> tag present (partial credit).
        - 0.0: Missing required tags.
        
    Args:
        prompts: List of input prompts from GRPOTrainer.
        completions: List of model completions.
        **kwargs: Additional dataset metadata.
        
    Returns:
        List[float]: Format compliance score (0.0, 0.5, or 1.0).
        
    Examples:
        >>> prompts = ['P1', 'P2']
        >>> comps = ['<reasoning>R</reasoning><answer>A</answer>', '<answer>A</answer>']
        >>> format_reward(prompts, comps)
        [1.0, 0.5]
    """
    rewards = []
    for completion in completions:
        text = extract_completion_text(completion)
        has_reasoning = bool(re.search(
            r"<reasoning>.*?</reasoning>", text, re.DOTALL
        ))
        has_answer = bool(re.search(
            r"<answer>.*?</answer>", text, re.DOTALL
        ))
        
        if has_reasoning and has_answer:
            rewards.append(1.0)
        elif has_answer:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards


print("Reward functions defined with comprehensive docstrings ✅")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Test the Reward Functions
# ═══════════════════════════════════════════════════════════════════

test_prompts = ["test"] * 5  # Dummy prompts for testing
test_completions = [
    # ✅ Perfect: correct answer + good format
    "<reasoning>\nJanet sells 16 - 3 - 4 = 9 eggs\n9 * 2 = 18\n</reasoning>\n<answer>18</answer>",
    
    # ❌ Good format but wrong answer
    "<reasoning>\n16 * 2 = 32\n</reasoning>\n<answer>32</answer>",
    
    # ⚠️ Missing reasoning tags
    "The answer is 18.\n<answer>18</answer>",
    
    # ❌ No tags at all
    "The answer is 18.",
    
    # ❌ Empty
    "",
]
test_solutions = ["18", "18", "18", "18", "18"]

c_rewards = correctness_reward(test_prompts, test_completions, test_solutions)
f_rewards = format_reward(test_prompts, test_completions)

print(f"{'Completion':<50} {'Correct':>8} {'Format':>8} {'Total':>8}")
print("-" * 78)
for comp, cr, fr in zip(test_completions, c_rewards, f_rewards):
    label = comp[:47] + "..." if len(comp) > 47 else comp if comp else "(empty)"
    print(f"{label:<50} {cr:>8.1f} {fr:>8.1f} {cr+fr:>8.1f}")

---
## 🧮 Section 3: Understanding GRPO

Before we train, let's understand what GRPO does at each step:

### The GRPO Training Loop

```
For each prompt:
  1. Generate G completions (G=4 in our case)
  2. Score each with reward functions → [r₁, r₂, r₃, r₄]
  3. Compute group-relative advantages:
     Â_i = (r_i - mean(r)) / std(r)
  4. Reinforce completions with positive advantage
     Discourage completions with negative advantage
```

### Key Insight
GRPO grades "on a curve" — a completion doesn't need to be perfect, it just needs to be better than the group average.

### Key Hyperparameters
| Parameter | Our Value | What It Controls |
|-----------|-----------|------------------|
| `num_generations` | 4 | Completions per prompt (G) |
| `beta` | 0.04 | KL penalty — keeps model close to original |
| `learning_rate` | 5e-6 | Update step size |
| `max_completion_length` | 512 | Max tokens per generated completion |

---
## 🔧 Section 4: Model Loading & LoRA Setup

In [ ]:
from unsloth import FastLanguageModel

# ═══════════════════════════════════════════════════════════════════
# Load Model (4-bit quantized to fit in T4 16GB VRAM)
# ═══════════════════════════════════════════════════════════════════

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024  # Prompt + completion max length

print(f"📦 Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # 4-bit quantization → ~1.2GB instead of ~6GB
    device_map="cuda:0", # Explicitly pin model to cuda:0
)
print(f"✅ Model loaded successfully on cuda:0!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Add LoRA Adapters
# ═══════════════════════════════════════════════════════════════════
# Instead of fine-tuning all 1.5B parameters (impossible on T4),
# we add small trainable LoRA matrices to key layers.
# Only ~1-5% of parameters become trainable.

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    lora_alpha=16,           # Scaling factor
    lora_dropout=0.05,       # Regularization
    target_modules=[         # Layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
        "gate_proj", "up_proj", "down_proj",        # MLP
    ],
    use_gradient_checkpointing="unsloth",  # Saves ~60% memory
)

# Report parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,} ({100*trainable_params/total_params:.2f}%)")

---
## 📊 Section 5: Pre-Training Baseline Evaluation

Before GRPO training, let's see how the base model performs on GSM8K.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Evaluate Base Model (Before GRPO)
# ═══════════════════════════════════════════════════════════════════

NUM_EVAL_SAMPLES = 20  # Quick eval on 20 samples

# Switch to inference mode
FastLanguageModel.for_inference(model)

base_correct = 0
base_format = 0
base_examples = []

print(f"📊 Evaluating base model on {NUM_EVAL_SAMPLES} test samples...\n")

for i in range(NUM_EVAL_SAMPLES):
    sample = test_dataset[i]
    messages = sample["prompt"]
    
    # Apply chat template
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            top_p=0.95,
            use_cache=True,
        )
    
    generated = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], 
        skip_special_tokens=True
    )
    
    # Check format
    has_fmt = bool(re.search(r"<reasoning>.*?</reasoning>.*<answer>.*?</answer>", generated, re.DOTALL))
    if has_fmt:
        base_format += 1
    
    # Check correctness
    is_correct = False
    match = re.search(r"<answer>\s*(.*?)\s*</answer>", generated, re.DOTALL)
    if match:
        ans = match.group(1).strip().replace(",", "").replace("$", "")
        try:
            is_correct = float(ans) == float(sample["solution"])
        except ValueError:
            is_correct = ans == sample["solution"]
    if is_correct:
        base_correct += 1
    
    base_examples.append({
        "question": messages[1]["content"][:80],
        "truth": sample["solution"],
        "generated": generated[:200],
        "correct": is_correct,
        "format": has_fmt,
    })
    print(f"  [{i+1:2d}/{NUM_EVAL_SAMPLES}] {'✅' if is_correct else '❌'} (fmt: {'✓' if has_fmt else '✗'}) truth={sample['solution']}")

print(f"\n{'='*50}")
print(f"BASE MODEL RESULTS:")
print(f"  Accuracy:   {base_correct}/{NUM_EVAL_SAMPLES} ({100*base_correct/NUM_EVAL_SAMPLES:.1f}%)")
print(f"  Format:     {base_format}/{NUM_EVAL_SAMPLES} ({100*base_format/NUM_EVAL_SAMPLES:.1f}%)")
print(f"{'='*50}")


In [ ]:
# Show a few examples from the base model
print("\n--- Base Model Example Outputs ---\n")
for ex in base_examples[:3]:
    print(f"Q: {ex['question']}...")
    print(f"Truth: {ex['truth']}")
    print(f"Model: {ex['generated'][:150]}...")
    print(f"Result: {'✅ Correct' if ex['correct'] else '❌ Wrong'}")
    print("-" * 60)

---
## 🚀 Section 6: GRPO Training

Now for the main event! We configure and launch GRPO training.

### What Happens During Training
```
Each step:
  1. Take 1 prompt
  2. Generate 4 completions (num_generations=4)
  3. Score all 4 with reward functions
  4. Compute group-relative advantages
  5. Update model weights via policy gradient
  
Repeat × 4 steps → accumulate gradients → apply update
Total effective batch = 4 prompts × 4 completions = 16 completions
```

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported

# ═══════════════════════════════════════════════════════════════════
# GRPO Training Configuration
# ═══════════════════════════════════════════════════════════════════

training_args = GRPOConfig(
    # Output directory
    output_dir="./grpo-qwen-gsm8k",
    
    # Training schedule
    num_train_epochs=1,
    max_steps=250,                     # 250 steps ≈ 2-3 hours on T4
    
    # Batch configuration
    per_device_train_batch_size=1,      # 1 prompt per step
    gradient_accumulation_steps=4,      # Accumulate over 4 prompts
    
    # GRPO-specific settings
    num_generations=4,                  # G: completions per prompt
    max_completion_length=512,          # Max tokens per completion
    
    # Optimizer
    learning_rate=5e-6,                 # Conservative for RL
    lr_scheduler_type="cosine",         # Cosine annealing
    warmup_ratio=0.1,                   # 10% warmup
    
    # KL penalty (keeps model close to original)
    beta=0.04,
    
    # Logging & saving
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,                 # Keep only 2 checkpoints
    
    # Precision (fp16 on T4, bf16 on Ampere/A100/L4)
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    gradient_checkpointing=True,
    
    # Reproducibility
    seed=42,
    report_to="none",                   # No WandB on Kaggle
)

print("Training config created ✅")
print(f"  Steps:          {training_args.max_steps}")
print(f"  Batch size:     {training_args.per_device_train_batch_size}")
print(f"  Grad accum:     {training_args.gradient_accumulation_steps}")
print(f"  Generations:    {training_args.num_generations}")
print(f"  Learning rate:  {training_args.learning_rate}")
print(f"  KL beta:        {training_args.beta}")
print(f"  Precision:      {'bfloat16' if training_args.bf16 else 'float16'}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Initialize GRPOTrainer
# ═══════════════════════════════════════════════════════════════════

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[correctness_reward, format_reward],
    processing_class=tokenizer,
)

print("GRPOTrainer initialized ✅")
print(f"  Reward functions: correctness_reward, format_reward")
print(f"  Training samples: {len(train_dataset)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 🚀 LAUNCH TRAINING
# ═══════════════════════════════════════════════════════════════════
# This will take approximately 2-3 hours on a T4 GPU.
# Watch for:
#   - reward/mean: should gradually increase
#   - reward/std: moderate values (0.2-0.8)
#   - kl: should stay low and stable (< 10)

print("🚀 Starting GRPO training...")
print("   This will take approximately 2-3 hours on T4.")
print("   Watch the reward/mean — it should gradually increase!")
print("=" * 60)

trainer.train()

print("=" * 60)
print("✅ Training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Save the Trained Model
# ═══════════════════════════════════════════════════════════════════

SAVE_PATH = "./grpo-qwen-gsm8k-final"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"💾 Model saved to {SAVE_PATH}")

# Show saved files
import os
for f in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(os.path.join(SAVE_PATH, f))
    print(f"  {f:<40} {size/1024:.1f} KB")

---
## 📈 Section 7: Post-Training Evaluation

Let's compare the trained model against the baseline we measured earlier.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Evaluate Trained Model (After GRPO)
# ═══════════════════════════════════════════════════════════════════

# Switch to inference mode
FastLanguageModel.for_inference(model)

trained_correct = 0
trained_format = 0
trained_examples = []

print(f"📊 Evaluating trained model on {NUM_EVAL_SAMPLES} test samples...\n")

for i in range(NUM_EVAL_SAMPLES):
    sample = test_dataset[i]
    messages = sample["prompt"]
    
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            top_p=0.95,
            use_cache=True,
        )
    
    generated = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], 
        skip_special_tokens=True
    )
    
    has_fmt = bool(re.search(r"<reasoning>.*?</reasoning>.*<answer>.*?</answer>", generated, re.DOTALL))
    if has_fmt:
        trained_format += 1
    
    is_correct = False
    match = re.search(r"<answer>\s*(.*?)\s*</answer>", generated, re.DOTALL)
    if match:
        ans = match.group(1).strip().replace(",", "").replace("$", "")
        try:
            is_correct = float(ans) == float(sample["solution"])
        except ValueError:
            is_correct = ans == sample["solution"]
    if is_correct:
        trained_correct += 1
    
    trained_examples.append({
        "question": messages[1]["content"][:80],
        "truth": sample["solution"],
        "generated": generated[:200],
        "correct": is_correct,
        "format": has_fmt,
    })
    print(f"  [{i+1:2d}/{NUM_EVAL_SAMPLES}] {'✅' if is_correct else '❌'} (fmt: {'✓' if has_fmt else '✗'}) truth={sample['solution']}")

print(f"\n{'='*50}")
print(f"TRAINED MODEL RESULTS:")
print(f"  Accuracy:   {trained_correct}/{NUM_EVAL_SAMPLES} ({100*trained_correct/NUM_EVAL_SAMPLES:.1f}%)")
print(f"  Format:     {trained_format}/{NUM_EVAL_SAMPLES} ({100*trained_format/NUM_EVAL_SAMPLES:.1f}%)")
print(f"{'='*50}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Before vs After Comparison
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("📊 COMPARISON: Base Model vs GRPO-Trained Model")
print("=" * 60)
print(f"")
print(f"{'Metric':<20} {'Before GRPO':>15} {'After GRPO':>15} {'Change':>10}")
print(f"{'-'*60}")

base_acc = 100 * base_correct / NUM_EVAL_SAMPLES
trained_acc = 100 * trained_correct / NUM_EVAL_SAMPLES
base_fmt = 100 * base_format / NUM_EVAL_SAMPLES
trained_fmt = 100 * trained_format / NUM_EVAL_SAMPLES

acc_change = trained_acc - base_acc
fmt_change = trained_fmt - base_fmt

print(f"{'Accuracy':<20} {base_acc:>14.1f}% {trained_acc:>14.1f}% {acc_change:>+9.1f}pp")
print(f"{'Format Compliance':<20} {base_fmt:>14.1f}% {trained_fmt:>14.1f}% {fmt_change:>+9.1f}pp")
print(f"")

if acc_change > 0:
    print(f"🎉 GRPO training improved accuracy by {acc_change:.1f} percentage points!")
elif acc_change == 0:
    print(f"🔄 Accuracy unchanged — try more training steps or different hyperparameters.")
else:
    print(f"⚠️  Accuracy decreased — check for reward hacking or training instability.")

if fmt_change > 10:
    print(f"📝 Format compliance significantly improved — model learned the XML structure!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Side-by-Side Example Comparison
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("SIDE-BY-SIDE EXAMPLES: Base vs Trained")
print("=" * 70)

for i in range(min(5, NUM_EVAL_SAMPLES)):
    base_ex = base_examples[i]
    trained_ex = trained_examples[i]
    
    print(f"\n{'─'*70}")
    print(f"Q: {base_ex['question']}...")
    print(f"Truth: {base_ex['truth']}")
    print(f"\n  BASE MODEL:    {'✅' if base_ex['correct'] else '❌'}")
    print(f"  {base_ex['generated'][:120]}...")
    print(f"\n  TRAINED MODEL: {'✅' if trained_ex['correct'] else '❌'}")
    print(f"  {trained_ex['generated'][:120]}...")

---
## 🎯 Section 8: Interactive Testing

Try your trained model on custom math problems!

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Try Your Own Math Problems
# ═══════════════════════════════════════════════════════════════════

def solve_math(question: str) -> str:
    """Ask the trained model to solve a math problem."""
    messages = format_prompt(question)
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            use_cache=True,
        )
    
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )


# Test problems
test_problems = [
    "A store sells pencils for $2 each and pens for $5 each. If Maria buys 3 pencils and 4 pens, how much does she spend in total?",
    "A train travels at 60 miles per hour for 2.5 hours. How many miles does it travel?",
    "Tom has 15 apples. He gives 3 to each of his 4 friends. How many apples does Tom have left?",
]

for problem in test_problems:
    print(f"\n{'='*60}")
    print(f"Q: {problem}")
    print(f"\nModel's response:")
    print(solve_math(problem))
    print()


---
## 📚 Summary & Next Steps

### What We Accomplished
1. ✅ Prepared GSM8K data with structured prompts
2. ✅ Built correctness and format reward verifiers
3. ✅ Trained Qwen2.5-1.5B with GRPO on a free T4 GPU
4. ✅ Evaluated improvement in math reasoning

### Key Takeaways
- **RLVR** uses deterministic verifiers instead of learned reward models
- **GRPO** generates multiple completions and "grades on a curve"
- Even small models (1.5B) can learn to reason with RLVR
- The reward function is the most important component to get right

### To Improve Results
- **More training steps**: 500-1000 instead of 250
- **More data**: Use full GSM8K (8K samples)
- **Larger model**: Qwen2.5-3B or 7B (may need T4x2)
- **Tune hyperparameters**: Learning rate, KL beta, group size

### Further Reading
- [DeepSeekMath Paper](https://arxiv.org/abs/2402.03300) — Original GRPO algorithm
- [DeepSeek-R1 Report](https://arxiv.org/abs/2501.12948) — GRPO at scale
- [TRL GRPOTrainer Docs](https://huggingface.co/docs/trl/main/en/grpo_trainer) — API reference
- [Unsloth Guide](https://docs.unsloth.ai/basics/reasoning) — Memory-efficient training